In [ ]:
import os
if 'experiments' in os.getcwd (): os.chdir (os.getcwd () + "/..")

####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/nnet_bagging.csv"

######## MOOOOO ##########
SEED = int ("".join (str (ord (c)) for c in 
            """  __________________
               < GORDOBOB GOT MILK! >
                 ------------------
                         \   ^__^
                          \  (oo)\________
                             (__)\        )\ 
                                 ||----w-||
                                 ||    ; ||
                      so much milk -> """)) % (2 ** 32)

# Preprocessing

In [7]:
TARGET_FEATURE =            "Milk_Yield_L"

DROP_FEATURES =            ["Cattle_ID",
                            "Farm_ID",
                            "Feed_Quantity_lb",
                            "Climate_Zone",
                            "Management_System",
                            "Feed_Type",
                            "Feeding_Frequency",
                            "Walking_Distance_km",
                            "Grazing_Duration_hrs",
                            "Rumination_Time_hrs",
                            "Resting_Hours",
                            "Humidity_percent",
                            "BVD_Vaccine",
                            "FMD_Vaccine",
                            "Brucellosis_Vaccine",
                            "HS_Vaccine",
                            "BQ_Vaccine",
                            "Housing_Score",
                            "Body_Condition_Score",
                            "Milking_Interval_hrs",
                            "Breed",
                            "Date"]

CATEGORICAL_FEATURES =     ["Season",
                            "Young",
                            "Lactation_Stage",
                            "IBR_Vaccine",
                            "Anthrax_Vaccine",
                            "Rabies_Vaccine"]

STANDARD_SCALED_FEATURES = ["Feed_Quantity_kg",
                            "Water_Intake_L",
                            "Parity",
                            "Ambient_Temperature_C",
                            "Previous_Week_Avg_Yield",
                            "Days_in_Milk",
                            "Age_Months",
                            "Weight_kg"]

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

def preprocess (
    dtrain, dtest = None, scaler = None
) -> tuple[pd.DataFrame, pd.DataFrame | None, StandardScaler]:
    """
    NOTES:
    - Interaction features do not help
    - Squaring feed for outlier overexageratting does not help
    - clipping negative records worsens results
    - Predictive and mean imputation worsen results
    - Robust & min-max scalers worsen results
    """
    dtrain = dtrain.copy()
    if dtest is not None:
        dtest = dtest.copy()
    
    ### CONVERT MONTH TO SEASON ###
    def month_to_season (m):
        if m in [12, 1, 2]:
            return "Winter"
        elif m in [3, 4, 5]:
            return "Spring"
        elif m in [6, 7, 8]:
            return "Summer"
        else:
            return "Fall"

    if dtest is not None:
        months = pd.to_datetime (dtest['Date']).dt.month
        dtest['Season'] = months.apply (month_to_season)


    months = pd.to_datetime (dtrain['Date']).dt.month
    dtrain['Season'] = months.apply (month_to_season)

    ### EXTRACT AGE/YOUTH PATTERN ###
    dtrain['Young'] = (dtrain['Age_Months'] < 60).astype (int)
    if dtest is not None:
        dtest['Young'] = (dtest['Age_Months'] < 60).astype (int)

    ### IMPUT MISSING FEED ###
    median_val = dtrain["Feed_Quantity_kg"].median ()

    dtrain.loc[dtrain["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val
    if dtest is not None:
        dtest.loc[dtest["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val

    ### ONE-HOT ENCODE CATEGORICALS ###
    dtrain = pd.get_dummies (dtrain, columns = CATEGORICAL_FEATURES, drop_first=False)
    if dtest is not None:
        dtest = pd.get_dummies (dtest, columns = CATEGORICAL_FEATURES, drop_first=False)
        dtrain, dtest = dtrain.align (dtest, join = 'left', axis = 1, fill_value = 0)

    ### STANDARDIZE DATA ###
    if scaler is None:
        scaler = StandardScaler ()
        dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform (dtrain[STANDARD_SCALED_FEATURES])
    else:
        dtrain[STANDARD_SCALED_FEATURES] = scaler.transform (dtrain[STANDARD_SCALED_FEATURES])
    
    if dtest is not None:
        dtest[STANDARD_SCALED_FEATURES] = scaler.transform (dtest[STANDARD_SCALED_FEATURES])
    
    ### DROP ALL THE USELESS FEATURES ###
    dtrain = dtrain.drop (DROP_FEATURES, axis = 1)
    if dtest is not None:
        dtest = dtest.drop (DROP_FEATURES, axis = 1)

    ### DONE ###
    return dtrain, dtest, scaler

# Training Set Eval

# Final Model

In [ ]:
from sklearn.neural_network import MLPRegressor

ITERATIONS_PER_STEP = 100
model_template = MLPRegressor (hidden_layer_sizes = (110, 110, 110),
                               activation = "tanh",
                               learning_rate_init = 0.00003,
                               learning_rate = "adaptive",
                               early_stopping = False, 
                               n_iter_no_change = 20,
                               verbose = False,
                               warm_start = False,
                               max_iter = ITERATIONS_PER_STEP,
                               random_state = SEED)

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings ('ignore', category = ConvergenceWarning)

# NOTE: Dropping negative records worsens results
train_data = pd.read_csv (TRAIN_PATH)
X_train = train_data.drop (TARGET_FEATURE, axis = 1)
Y_train = train_data[TARGET_FEATURE]
X_train_proc, _, scaler = preprocess(X_train, None)

bag_model = BaggingRegressor(
    estimator=model_template,
    n_estimators=10,
    bootstrap=True,
    max_features=1.0,
    oob_score=True,
    n_jobs=-1,
    random_state=SEED
)

print("Training bagged MLP...")
bag_model.fit(X_train_proc, Y_train)

oob_pred = bag_model.oob_prediction_
oob_rmse = np.sqrt(mean_squared_error(Y_train, oob_pred))
print(f"OOB RMSE (bagged MLP): {oob_rmse:.5f}")

train_pred = bag_model.predict(X_train_proc)
train_rmse = np.sqrt(mean_squared_error(Y_train, train_pred))
print(f"Train RMSE (bagged MLP): {train_rmse:.5f}")


Training fold 1/5...


/opt/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


  Iteration 10: Val RMSE = 4.19383


# Final Model

In [ ]:
print("Generating test predictions...")

# Load test data
X_test = pd.read_csv(TEST_PATH)

# Preprocess using the SAME scaler used for training
_, X_test_proc, _ = preprocess(X_train.copy(), X_test.copy(), scaler=scaler)

# Predict with the trained bagging ensemble
y_pred_ensemble = bag_model.predict(X_test_proc)

print(f"Ensemble prediction mean: {y_pred_ensemble.mean():.5f}")
print(f"Training label mean:      {Y_train.mean():.5f}")

# Build submission DataFrame
out_data = pd.DataFrame({
    "Cattle_ID": np.arange(1, len(y_pred_ensemble) + 1),
    "Milk_Yield_L": y_pred_ensemble
})

# Save predictions
out_data.to_csv(OUT_PATH, index=False)
print(f"Saved predictions to {OUT_PATH}")

Generating ensemble predictions...
Ensemble pred mean: 15.61414
Training label mean: 15.58916
